In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import mappy as mp
import pyranges as pr
import pandas as pd
from datasets import load_dataset
from Bio import SeqIO
from pathlib import Path
from intervaltree import IntervalTree
from collections import Counter
from src.utils.canonicalization import parse_transcribed_features_from_gff

# Paths
fasta_path = "../../data/test.mmi"
gff_path = "../../data/hg38/gencode.v50.annotation.gff3"

print("Loading data...")
aligner = mp.Aligner(fasta_path)
gr = pr.read_gff3(gff_path)
# Convert level to int to avoid comparison issues
gr.level = gr.level.astype(int)
print(f"Data loaded. GFF chroms: {list(gr.chromosomes)[:5]}...")

In [ ]:
dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")
task_name = "splice_sites_all" # "enhancers", "splice_sites_all", etc.
filtered = dataset.filter(lambda x: x["task"] == task_name)
print(f"Dataset filtered for {task_name}. Size: {len(filtered['train'])}")

In [ ]:
canonicalization_segments = {}  # chrom -> { 'positive': IntervalTree, 'negative': IntervalTree }

print("Processing GFF for canonicalization segments...")
gff_df = gr.df
for chrom, group in gff_df.groupby("Chromosome"):
    pos, neg = parse_transcribed_features_from_gff(
        group, chrom, annotations_level=2, include_cdjv_segments=True, include_pseudogenes=True
    )
    canonicalization_segments[chrom] = {
        "positive": IntervalTree.from_tuples(pos),
        "negative": IntervalTree.from_tuples(neg)
    }
print("Done.")

In [26]:
print(f"Running detailed loop for first 100 sequences...")
stats = Counter()
for i, seq in enumerate(filtered['train']['sequence'][:100]):
    stats['total'] += 1
    hit = next((x for x in aligner.map(seq) if x.is_primary), None)
    if hit is None:
        stats['no_hit'] += 1
        continue
        
    stats['has_hit'] += 1
    hit_id = hit.ctg
    if hit_id not in canonicalization_segments:
        stats['chrom_mismatch'] += 1
        continue

    segs = canonicalization_segments[hit_id]
    strand_key = 'positive' if hit.strand == 1 else 'negative'
    tree = segs[strand_key]

    if not tree.overlap(hit.r_st, hit.r_en):
        stats['miss'] += 1
        # Detailed diagnostic from the join (which was done without strand constraint)
        local_feats = df_diag[df_diag["hit_index"] == i]
        if not local_feats.empty:
            types = local_feats["Feature"].value_counts().to_dict()
            genes = local_feats["gene_name"].unique().tolist()
            # Strand_b is the strand of the feature in GFF (+ or -)
            hit_strand_str = "+" if hit.strand == 1 else "-"
            same_strand_count = (local_feats["Strand_b"] == hit_strand_str).sum()
            
            print(f"[MISS] {hit_id}:{hit.r_st}-{hit.r_en} strand={hit_strand_str} | "
                  f"types: {types} | genes: {genes} | same_strand: {same_strand_count}/{len(local_feats)}")
        else:
            print(f"[MISS] {hit_id}:{hit.r_st}-{hit.r_en} strand={'+' if hit.strand==1 else '-'} → intergenic")
    else:
        stats['overlap'] += 1

print(f"\nSummary: {dict(stats)}")

Running detailed loop for first 100 sequences...
[MISS] chr18:32412471-32412871 strand=+ → intergenic
[MISS] chr4:13333336-13333736 strand=+ → intergenic
[MISS] chr3:142817490-142817890 strand=+ → intergenic
[MISS] chr3:9882922-9883322 strand=+ → intergenic
[MISS] chr3:47470988-47471388 strand=+ → intergenic
[MISS] chr2:113066476-113066876 strand=+ → intergenic
[MISS] chr3:78247168-78247568 strand=+ → intergenic
[MISS] chr3:47942511-47942911 strand=+ → intergenic
[MISS] chr3:47679987-47680387 strand=+ → intergenic
[MISS] chr3:41041630-41042030 strand=+ → intergenic
[MISS] chr3:31463471-31463871 strand=+ → intergenic
[MISS] chr3:3205872-3206272 strand=+ → intergenic
[MISS] chr3:167725921-167726321 strand=+ → intergenic
[MISS] chr19:52452745-52453145 strand=+ → intergenic
[MISS] chr16:53219-53619 strand=+ → intergenic
[MISS] chr7:157441635-157442035 strand=+ → intergenic
[MISS] chr7:100728463-100728863 strand=+ → intergenic
[MISS] chr3:93931256-93931656 strand=+ → intergenic
[MISS] chr9: